# WasteLens - Iteration 12 (Colab GPU)


Compute env is Colab GPU; protocol unchanged from docs/rejection_experiment/iteration12_distillation_protocol.md (registered pre-training, commit 3b88e70). Run every code cell top to bottom; do not skip or reorder.

## S0 - environment

In [ ]:
!nvidia-smi


In [ ]:
import sys, platform
import tensorflow as tf
print('Python:', sys.version.split()[0], '|', platform.platform())
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'NONE - enable a GPU runtime')
assert gpus, 'STOP: no GPU visible'
import numpy, sklearn, PIL
print('numpy:', numpy.__version__,
      '| sklearn:', sklearn.__version__,
      '| pillow:', PIL.__version__)


## S1 - repo sync (start from main, pin registered commit)

In [ ]:
import subprocess
from pathlib import Path
REPO = Path('/content/WasteLens')
PINNED = 'd930d3c07ec183e1fa8fbf3c686488bac8922612'
if not (REPO / 'src' / 'train_distill_adaptation.py').exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/Anik00223/WasteLens.git',
                    str(REPO)], check=True)


In [ ]:
%cd /content/WasteLens
!git fetch origin main
!git checkout main
!git rev-parse HEAD
!git rev-parse origin/main
!git status --short


In [ ]:
import subprocess
head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                      text=True, cwd='/content/WasteLens').stdout.strip()
print('HEAD:', head)
assert head == 'd930d3c07ec183e1fa8fbf3c686488bac8922612', 'STOP: HEAD != pinned commit'
print('repo pinned OK')


## S2 - Drive setup + resume-aware dataset (thumb_url + SHA locks)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BK = '/content/drive/MyDrive/WasteLens/iteration12'
!mkdir -p "$BK/datasets/adaptation_set" "$BK/datasets/fresh_set" "$BK/checkpoints" "$BK/evidence" "$BK/logs"
print('backup root:', BK)


In [ ]:
%cd /content/WasteLens
!python src/materialize_manifest.py --manifest docs/rejection_experiment/adaptation_set_manifest.json --out scratch/adaptation_set --drive "$BK/datasets/adaptation_set" --report "$BK/datasets/adaptation_lock_report.json"
!python src/materialize_manifest.py --manifest docs/rejection_experiment/fresh_set_manifest.json --out scratch/fresh_set --drive "$BK/datasets/fresh_set" --report "$BK/datasets/fresh_lock_report.json"


In [ ]:
import sys
sys.path.insert(0, 'src')
import train_head_adaptation as th
print(th.lock_dataset())
import train_adaptation as wl10
print('shipped md5:', wl10.md5_file(th.SHIPPED_CHECKPOINT))
print('backup root BK =', BK)


## S3 - train seed 42 (6 epochs, FINAL epoch = candidate)

In [ ]:
%cd /content/WasteLens
!python -u src/train_distill_adaptation.py --seed 42 2>&1 | tee "$BK/logs/head12s42_train.log"


In [ ]:
%cd /content/WasteLens
!python src/backup_iter12.py save --drive "$BK" --seed 42
!ls -la models/checkpoints/*head12s42* docs/rejection_experiment/head12s42*
print('seed-42 artifacts mirrored to $BK')


## S4 - train seed 43 (identical recipe)

In [ ]:
%cd /content/WasteLens
!python -u src/train_distill_adaptation.py --seed 43 2>&1 | tee "$BK/logs/head12s43_train.log"


In [ ]:
%cd /content/WasteLens
!python src/backup_iter12.py save --drive "$BK" --seed 43
!ls -la models/checkpoints/*head12s43* docs/rejection_experiment/head12s43*
print('seed-43 artifacts mirrored to $BK')


## S5 - evaluate both seeds (threshold 0.0702, gates per protocol)

In [ ]:
%cd /content/WasteLens
!python -u src/eval_head_adaptation.py --seed 42 --tag head12 --protocol iteration12_distillation_protocol.md --gates head12_gates.json
!python -u src/eval_head_adaptation.py --seed 43 --tag head12 --protocol iteration12_distillation_protocol.md --gates head12_gates.json


In [ ]:
%cd /content/WasteLens
!python -u src/eval_head_adaptation.py --summary --tag head12 --protocol iteration12_distillation_protocol.md


## S6 - strict evidence bundle (repo + Drive, fail loudly if incomplete)

In [ ]:
%cd /content/WasteLens
!python src/package_head12_evidence.py --drive "$BK/evidence" --out head12_evidence.tar.gz
!cp head12_evidence.tar.gz "$BK/evidence/head12_evidence.tar.gz"
!ls -la head12_evidence.tar.gz "$BK/evidence/head12_evidence.tar.gz"
print('evidence in repo + $BK/evidence')


## S7 - SAVE CHECKPOINT NOW (manual: run any time before timeout)

In [ ]:
%cd /content/WasteLens
!python src/backup_iter12.py save --drive "$BK"
print('checkpoint saved to $BK')


## S8 - RESTORE FROM DRIVE (fresh runtime: datasets + artifacts)

In [ ]:
%cd /content/WasteLens
!python src/backup_iter12.py restore --drive "$BK"
!python src/materialize_manifest.py --manifest docs/rejection_experiment/adaptation_set_manifest.json --out scratch/adaptation_set --drive "$BK/datasets/adaptation_set" --retries 2
!python src/materialize_manifest.py --manifest docs/rejection_experiment/fresh_set_manifest.json --out scratch/fresh_set --drive "$BK/datasets/fresh_set" --retries 2
print('restore done; resume at the first missing S3/S4/S5 cell')


## S9 - DISASTER RECOVERY STATUS

In [ ]:
%cd /content/WasteLens
!echo "HEAD: $(git rev-parse HEAD)"; echo "origin: $(git rev-parse origin/main)"; git status --short
!ls scratch/adaptation_set | wc -l; !ls scratch/fresh_set | wc -l
!ls -la models/checkpoints/*head12s42*final.keras models/checkpoints/*head12s43*final.keras 2>/dev/null || echo 'NO head12 final checkpoints yet'
!ls -la docs/rejection_experiment/head12s42_gates.json docs/rejection_experiment/head12s43_gates.json docs/rejection_experiment/iteration12_decision.md 2>/dev/null || echo 'NO head12 eval/decision yet'
!ls -la head12_evidence.tar.gz "$BK/evidence/head12_evidence.tar.gz" 2>/dev/null || echo 'NO evidence archive yet'
!ls "$BK" 2>/dev/null || echo 'Drive backup root missing'
!python -c "import sys; sys.path.insert(0,'src'); import train_adaptation as w; print('shipped md5:', w.md5_file(w.SHIPPED_CHECKPOINT), '| threshold: 0.0702 (frozen)')"
